In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

processed_path = Path("../data/processed")

decision_data = pd.read_csv(
    processed_path / "finrisk_decision_intelligence.csv"
)

print("Dataset shape:", decision_data.shape)

display(decision_data.head())

Dataset shape: (10000, 26)


,customer_id,credit_risk_probability,credit_risk_score,temporal_risk_probability,temporal_risk_score,avg_fraud_probability,max_fraud_probability,avg_anomaly_score,max_anomaly_score,avg_hybrid_fraud_score,...,risk_category,calibrated_credit_probability,credit_uncertainty_score,credit_confidence_score,credit_uncertainty_category,human_review_required,decision,decision_reason,recommended_action,decision_confidence
0,100001,0.393545,39.354540,0.351935,35.193510,0.262764,0.783083,0.185125,0.816251,0.239472,...,Moderate,0.367032,0.734065,0.265935,High Uncertainty,True,Human Review,Elevated fraud risk; High model uncertainty,Escalate to human analyst for manual assessment,0.265935
1,100002,0.665469,66.546906,0.315314,31.531355,0.313527,0.803767,0.221139,0.709282,0.285810,...,Moderate,0.549241,0.901517,0.098483,High Uncertainty,True,Human Review,Elevated credit risk; High model uncertainty,Escalate to human analyst for manual assessment,0.098483
2,100003,0.515152,51.515220,0.410704,41.070427,0.476873,0.783061,0.261322,0.502814,0.412207,...,Moderate,0.451192,0.902384,0.097616,High Uncertainty,True,Human Review,Elevated credit risk; Elevated fraud risk; Hig...,Escalate to human analyst for manual assessment,0.097616
3,100004,0.411317,41.131725,0.357690,35.768967,0.266366,0.797005,0.227899,0.488348,0.254826,...,Moderate,0.364358,0.728716,0.271284,High Uncertainty,True,Human Review,High model uncertainty,Escalate to human analyst for manual assessment,0.271284
4,100005,0.573910,57.391030,0.325702,32.570225,0.295973,0.812917,0.247687,0.494577,0.281487,...,Moderate,0.434674,0.869349,0.130651,High Uncertainty,True,Human Review,Elevated credit risk; Elevated fraud risk; Hig...,Escalate to human analyst for manual assessment,0.130651


In [2]:
dashboard_summary = pd.DataFrame({
    "metric": [
        "Total Customers",
        "Average FinRisk Score",
        "High Risk Customers",
        "Critical Risk Customers",
        "Human Review Required",
        "Automated Approvals",
        "Enhanced Reviews",
        "Rejected Customers"
    ],
    "value": [
        len(decision_data),
        decision_data["finrisk_score"].mean(),
        (decision_data["finrisk_score"] >= 50).sum(),
        (decision_data["finrisk_score"] > 75).sum(),
        decision_data["human_review_required"].astype(bool).sum(),
        (decision_data["decision"] == "Approve").sum(),
        (decision_data["decision"] == "Enhanced Review").sum(),
        (decision_data["decision"] == "Reject").sum()
    ]
})

display(dashboard_summary)

,metric,value
0,Total Customers,10000.000000
1,Average FinRisk Score,36.727223
2,High Risk Customers,594.000000
3,Critical Risk Customers,0.000000
4,Human Review Required,8193.000000
5,Automated Approvals,1807.000000
6,Enhanced Reviews,0.000000
7,Rejected Customers,0.000000


In [3]:
dashboard_columns = [
    "customer_id",
    "finrisk_score",
    "risk_category",

    "credit_risk_score",
    "temporal_risk_score",
    "fraud_risk_score",
    "graph_risk_score",
    "financial_health_risk_score",

    "calibrated_credit_probability",
    "credit_uncertainty_score",
    "credit_confidence_score",
    "credit_uncertainty_category",

    "human_review_required",

    "decision",
    "decision_reason",
    "recommended_action"
]

dashboard_data = decision_data[
    dashboard_columns
].copy()

print("Dashboard dataset:", dashboard_data.shape)

display(dashboard_data.head())

Dashboard dataset: (10000, 16)


,customer_id,finrisk_score,risk_category,credit_risk_score,temporal_risk_score,fraud_risk_score,graph_risk_score,financial_health_risk_score,calibrated_credit_probability,credit_uncertainty_score,credit_confidence_score,credit_uncertainty_category,human_review_required,decision,decision_reason,recommended_action
0,100001,38.242348,Moderate,39.354540,35.193510,51.192378,20.645161,35.024149,0.367032,0.734065,0.265935,High Uncertainty,True,Human Review,Elevated fraud risk; High model uncertainty,Escalate to human analyst for manual assessment
1,100002,46.359409,Moderate,66.546906,31.531355,48.650618,29.582111,34.890943,0.549241,0.901517,0.098483,High Uncertainty,True,Human Review,Elevated credit risk; High model uncertainty,Escalate to human analyst for manual assessment
2,100003,45.627977,Moderate,51.515220,41.070427,53.346915,27.661290,44.734034,0.451192,0.902384,0.097616,High Uncertainty,True,Human Review,Elevated credit risk; Elevated fraud risk; Hig...,Escalate to human analyst for manual assessment
3,100004,40.674674,Moderate,41.131725,35.768967,48.225006,41.500701,29.000047,0.364358,0.728716,0.271284,High Uncertainty,True,Human Review,High model uncertainty,Escalate to human analyst for manual assessment
4,100005,43.396794,Moderate,57.391030,32.570225,53.410559,12.903226,43.773144,0.434674,0.869349,0.130651,High Uncertainty,True,Human Review,Elevated credit risk; Elevated fraud risk; Hig...,Escalate to human analyst for manual assessment


In [4]:
customers = pd.read_csv(
    "../data/raw/customers.csv"
)

customers["customer_id"] = (
    customers["customer_id"]
    .astype(str)
    .str.strip()
)

dashboard_data["customer_id"] = (
    dashboard_data["customer_id"]
    .astype(str)
    .str.strip()
)

demographic_columns = [
    "customer_id",
    "age",
    "employment_type",
    "annual_income",
    "employment_years",
    "dependents",
    "savings",
    "existing_debt",
    "credit_score"
]

dashboard_data = dashboard_data.merge(
    customers[demographic_columns],
    on="customer_id",
    how="left"
)

print("Final dashboard dataset:", dashboard_data.shape)

display(dashboard_data.head())

Final dashboard dataset: (10000, 24)


,customer_id,finrisk_score,risk_category,credit_risk_score,temporal_risk_score,fraud_risk_score,graph_risk_score,financial_health_risk_score,calibrated_credit_probability,credit_uncertainty_score,...,decision_reason,recommended_action,age,employment_type,annual_income,employment_years,dependents,savings,existing_debt,credit_score
0,100001,38.242348,Moderate,39.354540,35.193510,51.192378,20.645161,35.024149,0.367032,0.734065,...,Elevated fraud risk; High model uncertainty,Escalate to human analyst for manual assessment,59,Contract,6.542970e+05,0,3,281262.935425,185332.880592,708
1,100002,46.359409,Moderate,66.546906,31.531355,48.650618,29.582111,34.890943,0.549241,0.901517,...,Elevated credit risk; High model uncertainty,Escalate to human analyst for manual assessment,49,Salaried,6.216075e+05,4,3,189110.449367,116700.181303,810
2,100003,45.627977,Moderate,51.515220,41.070427,53.346915,27.661290,44.734034,0.451192,0.902384,...,Elevated credit risk; Elevated fraud risk; Hig...,Escalate to human analyst for manual assessment,35,Salaried,6.371762e+05,10,2,396507.236148,67320.829790,595
3,100004,40.674674,Moderate,41.131725,35.768967,48.225006,41.500701,29.000047,0.364358,0.728716,...,High model uncertainty,Escalate to human analyst for manual assessment,63,Salaried,1.915457e+06,19,4,350710.898806,217865.937714,761
4,100005,43.396794,Moderate,57.391030,32.570225,53.410559,12.903226,43.773144,0.434674,0.869349,...,Elevated credit risk; Elevated fraud risk; Hig...,Escalate to human analyst for manual assessment,28,Business Owner,3.833640e+05,10,4,324423.445016,446028.336260,689


In [5]:
dashboard_file = (
    processed_path /
    "finrisk_dashboard_data.csv"
)

dashboard_data.to_csv(
    dashboard_file,
    index=False
)

print("✓ Saved:")
print(dashboard_file)

✓ Saved:
..\data\processed\finrisk_dashboard_data.csv


In [6]:
dashboard_kpis = {
    "total_customers": int(len(dashboard_data)),

    "average_risk": round(
        dashboard_data["finrisk_score"].mean(),
        2
    ),

    "high_risk_customers": int(
        (dashboard_data["finrisk_score"] >= 50).sum()
    ),

    "critical_risk_customers": int(
        (dashboard_data["finrisk_score"] > 75).sum()
    ),

    "human_review": int(
        dashboard_data["human_review_required"]
        .astype(bool)
        .sum()
    ),

    "approval_count": int(
        (dashboard_data["decision"] == "Approve").sum()
    ),

    "review_count": int(
        (
            dashboard_data["decision"]
            .isin(["Enhanced Review", "Human Review"])
        ).sum()
    ),

    "rejection_count": int(
        (dashboard_data["decision"] == "Reject").sum()
    )
}

dashboard_kpis

{'total_customers': 10000,
 'average_risk': np.float64(36.73),
 'high_risk_customers': 594,
 'critical_risk_customers': 0,
 'human_review': 8193,
 'approval_count': 1807,
 'review_count': 8193,
 'rejection_count': 0}